<a href="https://colab.research.google.com/github/CodewithSaira/ML-Pipelining/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CodewithSaira/ML-Pipelining/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis (Grain):**
One row represents a unique combination of a search query, a page URL, and a specific date (`query` + `page` + `date`).

**Time Window:**
Mid-panel month `2026-03` (March 1, 2026 to March 31, 2026). The final month `2026-06` is kept strictly sealed for out-of-time evaluation.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**1. Features:**
* `impressions`: Total search result impressions (knowable at decision time).
* `position`: Average rank position on Google SERP (knowable at decision time).
* `query_length`: Character length of the search query (knowable at decision time).
* `word_count`: Total words in the search query (knowable at decision time).
* `is_branded`: Boolean flag indicating if query contains core brand terms (knowable at decision time).

**2. Label (Target):**
* `is_clicked`: Binary indicator (`1` if `clicks > 0` else `0`).

**3. Context Fields:**
* `date`, `query`, `page`: Join keys and partition granularity identifiers.

**4. Excluded Fields (and Why):**
* `ctr` (Click-Through Rate) and `clicks`: Excluded as direct features to prevent target leakage, since `clicks` directly defines the target label `is_clicked`.
* Low volume rows (`impressions < 5`): Excluded to remove noisy, non-representative long-tail search records.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
# Install and import required packages
!pip install duckdb datasets -q
import duckdb
from datasets import load_dataset
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. Load Dataset using HF_TOKEN from Secrets
hf_token = userdata.get('HF_TOKEN')
dataset = load_dataset("FlyRank/internship-warehouse", "fact_content_query_90d", token=hf_token)

# Convert split to pandas dataframe
df = dataset['train'].to_pandas() if isinstance(dataset, dict) and 'train' in dataset else dataset.to_pandas()

# Exact Mapped Columns from Output
imp_col = 'impressions_90d'
click_col = 'clicks_90d'
pos_col = 'avg_position_90d'
query_col = 'query_hash_id'
client_col = 'client_hash_id'

# 2. Prepare df_clean (Filtering & Feature Engineering)
df_clean = df[df[imp_col] >= 5].copy()
df_clean['is_clicked'] = (df_clean[click_col] > 0).astype(int)
df_clean['query_length'] = df_clean[query_col].astype(str).str.len()
df_clean['word_count'] = df_clean[query_col].astype(str).str.split().str.len()

# 3. Query 1: Prove Grain Uniqueness using DuckDB SQL
con = duckdb.connect()
con.register('df_table', df_clean)

grain_check = con.execute(f"""
    SELECT {query_col}, {client_col}, COUNT(*) as cnt
    FROM df_table
    GROUP BY {query_col}, {client_col}
    HAVING COUNT(*) > 1
""").fetchall()

print("1. Grain Uniqueness Check (Duplicates count):", len(grain_check))

# 4. Query 2: Row Count Verification
counts_check = con.execute(f"""
    SELECT
        COUNT(*) as total_rows,
        MIN({client_col}) as sample_client_min,
        MAX({client_col}) as sample_client_max
    FROM df_table
""").df()

print("\n2. Counts & ID Bounds:")
print(counts_check)

# 5. Query 3: Availability Verification with IS TRUE
avail_check = con.execute(f"""
    SELECT COUNT(*) as available_rows
    FROM df_table
    WHERE ({imp_col} IS NOT NULL AND {pos_col} IS NOT NULL) IS TRUE
""").fetchone()[0]

print("\n3. Available valid rows (IS TRUE check):", f"{avail_check:,}")

# --- THE LEAKAGE TRAP EXPERIMENT (Fast Sampled ML Training) ---
# Fast sub-sampling (100k rows) so training finishes in 3 seconds without crash
df_ml = df_clean.sample(n=min(100000, len(df_clean)), random_state=42)

# Feature set WITH Leaked Feature (clicks)
X_leaked = df_ml[[imp_col, pos_col, 'query_length', 'word_count', click_col]]
y = df_ml['is_clicked']

clf_leaked = RandomForestClassifier(n_estimators=10, random_state=42, n_jobs=-1)
clf_leaked.fit(X_leaked, y)
score_leaked = roc_auc_score(y, clf_leaked.predict_proba(X_leaked)[:, 1])
print(f"\n[LEAKAGE TRAP] ROC-AUC with leaked '{click_col}' feature: {score_leaked:.4f}")

# Feature set HONEST (clicks removed)
X_honest = df_ml[[imp_col, pos_col, 'query_length', 'word_count']]
clf_honest = RandomForestClassifier(n_estimators=10, random_state=42, n_jobs=-1)
clf_honest.fit(X_honest, y)
score_honest = roc_auc_score(y, clf_honest.predict_proba(X_honest)[:, 1])
print(f"[HONEST MODEL] ROC-AUC without leaked feature: {score_honest:.4f}")

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2414248 [00:00<?, ? examples/s]

1. Grain Uniqueness Check (Duplicates count): 430029

2. Counts & ID Bounds:
   total_rows        sample_client_min        sample_client_max
0     2414248  client_06d356715a8ff3b6  client_ff644d8251367cbb

3. Available valid rows (IS TRUE check): 2,414,248

[LEAKAGE TRAP] ROC-AUC with leaked 'clicks_90d' feature: 1.0000
[HONEST MODEL] ROC-AUC without leaked feature: 0.9800


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named Limitation:**
Search Console data is subject to privacy thresholding and sampling limits by Google (anonymized queries with low volume are omitted). Consequently, this slice cannot capture 100% of zero-click long-tail search traffic, causing an observed truncation bias towards higher-impression search terms.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.